<pre>
<center><b><h1>Loan Default Prediction</b></center>
<center><b><h1>Week - 5 | Model Evaluation & Tuning</b></center>
<pre>

### Step 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print('All libraries imported successfully!')

C:\Users\rkgaj\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


All libraries imported successfully!


### Step 2. Load & Prepare Dataset

In [2]:
df = pd.read_csv(r'C:\R drive\Sem-5\ML\MLProject\Loan_default.csv')
print('Shape:', df.shape)
df.head(3)

Shape: (255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1


### Step 3. Separate Features (X) and Target (y)

In [3]:
# Drop identifier and target from features
X = df.drop(['Default', 'LoanID'], axis=1)
y = df['Default']  # Already 0/1

print('X shape:', X.shape)
print('y distribution:')
print(y.value_counts())

X shape: (255347, 16)
y distribution:
Default
0    225694
1     29653
Name: count, dtype: int64


### Step 4. Identify and Encode Categorical Columns

In [4]:
# Binary Yes/No columns
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    X[col] = X[col].map({'Yes': 1, 'No': 0})

# Multi-category columns
multi_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']
X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

# Ensure all are int
bool_cols = X.select_dtypes(include='bool').columns.tolist()
X[bool_cols] = X[bool_cols].astype(int)

print('Encoded X shape:', X.shape)
print('Columns:', X.columns.tolist())

Encoded X shape: (255347, 24)
Columns: ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'HasMortgage', 'HasDependents', 'HasCoSigner', 'Education_High School', "Education_Master's", 'Education_PhD', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Married', 'MaritalStatus_Single', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other']


### Step 5. Split into Train and Test Sets

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)

Train size: (204277, 24)
Test size : (51070, 24)


### Step 6. Scale Numerical Features

In [6]:
numeric_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore',
                'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print('Scaling complete.')

Scaling complete.


---
## ✅ Tasks 1, 2, 3 & 4 — Model Evaluation & Comparison

**Covers:**
- Task 1: Accuracy, Precision, Recall, F1-score for each model
- Task 2: Train vs Test score (Overfitting check)
- Task 3: 5-Fold Cross-Validation
- Task 4: Comparison table — best model selection

In [7]:
# Define all 5 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []

for name, clf in models.items():
    print(f'Training {name}...', end=' ', flush=True)
    
    # Use scaled data only for Logistic Regression; tree models work fine without scaling
    if name == 'Logistic Regression':
        Xtr, Xte = X_train_scaled, X_test_scaled
    else:
        Xtr, Xte = X_train, X_test
    
    # Fit
    clf.fit(Xtr, y_train)
    
    # Predictions
    y_pred = clf.predict(Xte)
    
    # Task 1 — Evaluation metrics
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall    = recall_score(y_test, y_pred, zero_division=0)
    f1        = f1_score(y_test, y_pred, zero_division=0)
    
    # Task 2 — Overfitting check
    train_acc = clf.score(Xtr, y_train)
    diff = train_acc - acc
    if train_acc > 0.95 and diff > 0.08:
        fit_status = 'Overfitting ⚠️'
    elif train_acc < 0.70 and acc < 0.70:
        fit_status = 'Underfitting 🔴'
    else:
        fit_status = 'Good Fit ✅'
    
    # Task 3 — 5-Fold Cross Validation
    cv_scores = cross_val_score(clf, Xtr, y_train, cv=5, scoring='accuracy')
    cv_mean   = cv_scores.mean()
    cv_std    = cv_scores.std()
    
    results.append({
        'Model':           name,
        'Train Acc':       round(train_acc, 4),
        'Test Acc (Accuracy)': round(acc, 4),
        'Precision':       round(precision, 4),
        'Recall':          round(recall, 4),
        'F1-Score':        round(f1, 4),
        '5-Fold CV Avg':   round(cv_mean, 4),
        'CV Spread (Std)': round(cv_std, 4),
        'Fit Status':      fit_status,
    })
    print('Done!')

print('\nAll models trained!')

Training Logistic Regression... 

Done!
Training Decision Tree... 

Done!
Training Random Forest... 

Done!
Training AdaBoost... 

Done!
Training Gradient Boosting... 

Done!

All models trained!


In [8]:
# Task 4 — Comparison Table
results_df = pd.DataFrame(results).set_index('Model')
print('=== Model Comparison Table ===')
print(results_df.to_string())
results_df

=== Model Comparison Table ===
                     Train Acc  Test Acc (Accuracy)  Precision  Recall  F1-Score  5-Fold CV Avg  CV Spread (Std)      Fit Status
Model                                                                                                                           
Logistic Regression     0.6752               0.6764     0.2196  0.6997    0.3343         0.6752           0.0006  Underfitting 🔴
Decision Tree           1.0000               0.8019     0.1973  0.2301    0.2125         0.8029           0.0005  Overfitting ⚠️
Random Forest           1.0000               0.8853     0.6263  0.0305    0.0582         0.8855           0.0002  Overfitting ⚠️
AdaBoost                0.8858               0.8857     0.6110  0.0432    0.0806         0.8858           0.0003      Good Fit ✅
Gradient Boosting       0.8869               0.8864     0.6366  0.0511    0.0946         0.8863           0.0004      Good Fit ✅


,Train Acc,Test Acc (Accuracy),Precision,Recall,F1-Score,5-Fold CV Avg,CV Spread (Std),Fit Status
Model,,,,,,,,
Logistic Regression,0.6752,0.6764,0.2196,0.6997,0.3343,0.6752,0.0006,Underfitting 🔴
Decision Tree,1.0000,0.8019,0.1973,0.2301,0.2125,0.8029,0.0005,Overfitting ⚠️
Random Forest,1.0000,0.8853,0.6263,0.0305,0.0582,0.8855,0.0002,Overfitting ⚠️
AdaBoost,0.8858,0.8857,0.6110,0.0432,0.0806,0.8858,0.0003,Good Fit ✅
Gradient Boosting,0.8869,0.8864,0.6366,0.0511,0.0946,0.8863,0.0004,Good Fit ✅


In [9]:
# Pick best model: highest 5-Fold CV Avg with low CV Spread
best_idx   = results_df['5-Fold CV Avg'].idxmax()
best_row   = results_df.loc[best_idx]
print(f'🏆 Best Model Selected: {best_idx}')
print(f"   5-Fold CV Avg  : {best_row['5-Fold CV Avg']}")
print(f"   CV Spread (Std): {best_row['CV Spread (Std)']}")
print(f"   Reason: Highest Cross-Validation accuracy with lowest spread = most stable model")

🏆 Best Model Selected: Gradient Boosting
   5-Fold CV Avg  : 0.8863
   CV Spread (Std): 0.0004
   Reason: Highest Cross-Validation accuracy with lowest spread = most stable model


---
## ✅ Task 5 — Hyperparameter Tuning with GridSearchCV

Apply GridSearchCV to the **best model** (Random Forest) to find optimal parameters.

In [10]:
print('Running GridSearchCV on Random Forest...')
print('(This may take a few minutes)')

param_grid = {
    'n_estimators': [50, 100],
    'max_depth':    [None, 10],
}

rf_base = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print('\n=== GridSearchCV Results ===')
print('Best Parameters Found:', grid_search.best_params_)
print('Best CV Score        :', round(grid_search.best_score_, 4))

Running GridSearchCV on Random Forest...
(This may take a few minutes)
Fitting 3 folds for each of 4 candidates, totalling 12 fits



=== GridSearchCV Results ===
Best Parameters Found: {'max_depth': None, 'n_estimators': 100}
Best CV Score        : 0.8853


In [11]:
# Evaluate tuned model on test set
best_rf = grid_search.best_estimator_
tuned_pred = best_rf.predict(X_test)
tuned_acc  = accuracy_score(y_test, tuned_pred)

# Baseline (default RF)
baseline_acc = results_df.loc['Random Forest', 'Test Acc (Accuracy)']

print(f'Tuned Model Test Accuracy  : {tuned_acc:.4f}')
print(f'Baseline Test Accuracy was : {baseline_acc}')
print(f'Improvement                : {tuned_acc - baseline_acc:+.4f}')

Tuned Model Test Accuracy  : 0.8853
Baseline Test Accuracy was : 0.8853
Improvement                : -0.0000


---
## ✅ Task 6 — Advanced Models Implemented

All three advanced ensemble models have been trained above:

- ✅ **Random Forest** (Bagging)
- ✅ **AdaBoost** (Boosting)
- ✅ **Gradient Boosting** (Advanced Boosting)

### 🧠 Understanding the Models: Which to choose and why?

**1. Logistic Regression:** A basic statistical baseline model. It calculates the probability of default using a linear mathematical equation. It is very fast and easy to interpret, but struggles if the relationships in the data are complex or non-linear.

**2. Decision Tree:** A flowchart-like model that splits data based on questions (e.g., "Is income < $50k?"). It's highly interpretable but notoriously prone to *overfitting* — meaning it tends to memorize the training data rather than learning general rules.

**3. Random Forest (Bagging):** An ensemble method that builds hundreds of different Decision Trees and averages their predictions (majority vote). **Why choose it:** It automatically fixes the overfitting problem of single Decision Trees. It is extremely robust, stable, and usually performs exceptionally well right out of the box.

**4. AdaBoost (Boosting):** Instead of building trees independently, AdaBoost builds them sequentially. Each new tree focuses specifically on correcting the mistakes (misclassifications) made by the previous tree. **Why choose it:** It is excellent at boosting accuracy on borderline or difficult-to-predict applicants.

**5. Gradient Boosting (Advanced Boosting):** Similar to AdaBoost, but uses an advanced mathematical technique (gradient descent) to minimize prediction errors. **Why choose it:** It frequently yields the absolute highest accuracy in machine learning competitions, though it requires careful hyperparameter tuning to avoid overfitting.

In [12]:
print('=== Task 6 Summary: Advanced Models ===')
print()
for name in ['Random Forest', 'AdaBoost', 'Gradient Boosting']:
    row = results_df.loc[name]
    print(f'✅ {name}')
    print(f'   Test Accuracy : {row["Test Acc (Accuracy)"]}')
    print(f'   F1-Score      : {row["F1-Score"]}')
    print(f'   5-Fold CV     : {row["5-Fold CV Avg"]} ± {row["CV Spread (Std)"]}')
    print(f'   Fit Status    : {row["Fit Status"]}')
    print()

=== Task 6 Summary: Advanced Models ===

✅ Random Forest
   Test Accuracy : 0.8853
   F1-Score      : 0.0582
   5-Fold CV     : 0.8855 ± 0.0002
   Fit Status    : Overfitting ⚠️

✅ AdaBoost
   Test Accuracy : 0.8857
   F1-Score      : 0.0806
   5-Fold CV     : 0.8858 ± 0.0003
   Fit Status    : Good Fit ✅

✅ Gradient Boosting
   Test Accuracy : 0.8864
   F1-Score      : 0.0946
   5-Fold CV     : 0.8863 ± 0.0004
   Fit Status    : Good Fit ✅



---
## 💡 Why did we select Random Forest as the winner?

The system automatically selected **Random Forest** because it achieved the highest **5-Fold Cross-Validation Average Score**.

**Why does Cross-Validation matter?** Instead of just testing the model once, Cross-Validation cuts the data into 5 equal pieces. It trains the model 5 separate times on different chunks and averages the score. This proves that the model didn't just get "lucky" on one specific test set.

A high Cross-Validation score, combined with a low CV Spread (Standard Deviation), guarantees that the model is both highly accurate and highly stable, making it the safest choice for deploying into a real-world Loan Prediction Engine.

In [13]:
# Save evaluation results for the frontend to consume
evaluation_data = {
    'models': results_df.reset_index().to_dict('records'),
    'best_model': best_idx,
    'best_cv_score': float(best_row['5-Fold CV Avg']),
    'tuning': {
        'best_params': grid_search.best_params_,
        'tuned_accuracy': round(float(tuned_acc), 4),
        'baseline_accuracy': float(baseline_acc)
    }
}

output_path = r'C:\R drive\Sem-5\ML\MLProject\backend\evaluation.json'
with open(output_path, 'w') as f:
    json.dump(evaluation_data, f, indent=2)

print(f'Evaluation data saved to: {output_path}')
print('Run this cell after training to update the frontend Task 5 page!')

Evaluation data saved to: C:\R drive\Sem-5\ML\MLProject\backend\evaluation.json
Run this cell after training to update the frontend Task 5 page!
